# Chapter 23 — Relational data modeling and SQL for GUI analytics

**Book role:** Databases  
**Suggested book space:** about 9 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Explain tables, rows, keys, relationships, and normalization at a practical level.
- Write parameterized SELECT, JOIN, GROUP BY, and transaction operations.
- Select only necessary columns and rows for a GUI workflow.
- Map relational query results into a stable DataFrame schema.

## 1. Model the business grain before querying

A relational table represents a defined kind of fact or entity. An `orders` table may represent one row per order; an `order_items` table may represent one row per product line. Confusing these grains creates double-counted revenue, inflated order counts, and incorrect averages. Write down what one row means and which columns identify it before designing a query or dashboard.

Primary keys uniquely identify rows; foreign keys express relationships. Constraints can prevent invalid references and duplicates at the storage boundary. Normalization reduces unnecessary duplication, while analytical queries often join normalized operational tables into a reporting view. For a course-sized app, favor clear names, stable keys, and documented business definitions over sophisticated schema tricks.

### Worked example 1: Join and aggregate order data

The date range is half-open: start is included and end is excluded. This avoids boundary overlap when consecutive reports use adjacent date ranges. Check how joins affect row counts before aggregating.

In [ ]:
sql = """
SELECT
    o.region,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(i.quantity * i.unit_price) AS gross_sales
FROM orders AS o
JOIN order_items AS i ON i.order_id = o.order_id
WHERE o.order_date >= %s AND o.order_date < %s
GROUP BY o.region
ORDER BY gross_sales DESC
"""
print(sql)
# Parameter placeholder syntax is driver-specific. MySQL Connector/Python uses %s.
# Always execute with bound parameters, e.g. cursor.execute(sql, (start_date, end_date)).

## 2. Read queries and parameterization

Use SELECT statements that enumerate needed columns rather than `SELECT *` in a stable GUI contract. Filter rows in SQL when the database contains many records, use indexes that support actual filter patterns, and sort by a deterministic order when paginating. `GROUP BY` and aggregate functions can reduce network transfer and memory when only summaries are required.

Always bind values as query parameters through the database driver or SQLAlchemy. Do not interpolate a user-entered search string into SQL. A GUI control can limit possible values, but the application still must use parameterized SQL. Table and column identifiers usually cannot be bound as ordinary parameters; choose them from a trusted allow-list if users can select a field or sort direction.

### Worked example 2: A repository contract for relational results

A protocol describes expected behavior without forcing one specific database implementation. A CSV adapter and a MySQL adapter can both fulfill the contract if they return the agreed schema.

In [ ]:
from typing import Protocol
import pandas as pd

class SalesRepository(Protocol):
    def fetch_sales(self, start_date: str, end_date: str, region: str | None = None) -> pd.DataFrame:
        """Return normalized line-level sales for a half-open date range."""
        ...

REQUIRED_COLUMNS = {"order_id", "order_date", "region", "quantity", "unit_price"}

def validate_repository_frame(frame: pd.DataFrame) -> pd.DataFrame:
    missing = REQUIRED_COLUMNS - set(frame.columns)
    if missing:
        raise ValueError(f"Repository result missing columns: {sorted(missing)}")
    result = frame.copy()
    result["order_date"] = pd.to_datetime(result["order_date"], errors="raise")
    result["quantity"] = pd.to_numeric(result["quantity"], errors="raise")
    result["unit_price"] = pd.to_numeric(result["unit_price"], errors="raise")
    return result

## 3. Transactions, consistency, and pagination

A transaction groups operations so the database can commit or roll them back according to its guarantees. For write workflows, validate data first, begin the transaction using the driver/engine API, execute parameterized statements, and commit only when the operation succeeds. On error, roll back and show the user a failure rather than pretending that the update succeeded.

Pagination improves interactive response time, but OFFSET-based pagination may become costly for large tables and can shift when concurrent rows change. Use deterministic ordering, and consider keyset pagination when scale and requirements justify it. Explain whether a displayed result represents a snapshot or a fresh query; an analytics app should not imply transactional consistency that it does not have.

## Guided lab: Lab: database-backed filter contract

Define a relational model for customers, orders, order items, and products. State the grain and keys for every table. Write queries for a date filter, a parameterized region filter, a region summary, and a paginated detail list. Check the aggregate against a manually computed tiny fixture to expose join multiplication.

## Review questions and practice

1. Why should the data grain be specified before metrics are written?
2. What is a half-open date range and why is it useful?
3. Why is parameterized SQL required even with constrained GUI widgets?
4. What role does a foreign key play?
5. Why does pagination require stable ordering?

## Further reading

- MySQL SELECT: https://dev.mysql.com/doc/refman/8.4/en/select.html
- Python DB-API concepts: https://peps.python.org/pep-0249/

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.